In [11]:
from acfx import AcfxEBM
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

# Initialize model and explainer
model = ExplainableBoostingClassifier()
explainer = AcfxEBM(model)

# 1) Load
data = pd.read_csv("data/acfx_table.csv")  # adjust path if needed

# 2) Target + features
y = data["cat_value"]
X = data.drop(columns=["cat_value"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [12]:
# 3) Optuna-derived spec (ranges) + which columns are discrete/encoded
SPEC = {
    # encoded categoricals / discrete domains
    "params_booster": {"type": "cat"},
    "params_grow_policy": {"type": "cat"},
    "params_sample_type": {"type": "cat"},         # dart-only
    "params_normalize_type": {"type": "cat"},      # dart-only
    "user_attrs_resampling_strategy": {"type": "cat"},
    "user_attrs_missing_rate": {"type": "cat"},    # discretized grid
    "soft_risk_preference": {"type": "cat"},
    "soft_decision_speed": {"type": "cat"},

    # numeric floats
    "params_learning_rate": {"type": "float", "low": 1e-4, "high": 0.5, "scale": "log"},
    "params_min_child_weight": {"type": "float", "low": 0.1, "high": 100.0, "scale": "log"},
    "params_lambda": {"type": "float", "low": 1e-4, "high": 100.0, "scale": "log"},
    "params_alpha": {"type": "float", "low": 1e-4, "high": 100.0, "scale": "log"},
    "params_gamma": {"type": "float", "low": 0.0, "high": 10.0, "scale": "linear"},
    "params_subsample": {"type": "float", "low": 0.2, "high": 1.0, "scale": "linear"},
    "params_colsample_bytree": {"type": "float", "low": 0.2, "high": 1.0, "scale": "linear"},
    "params_rate_drop": {"type": "float", "low": 0.0, "high": 0.5, "scale": "linear"},     # dart-only
    "params_skip_drop": {"type": "float", "low": 0.0, "high": 0.9, "scale": "linear"},     # dart-only
    "user_attrs_sample_rows": {"type": "float", "low": 0.5, "high": 1.0, "scale": "linear"},
    "user_attrs_feature_fraction": {"type": "float", "low": 0.5, "high": 1.0, "scale": "linear"},
    "user_attrs_labeled_ratio": {"type": "float", "low": 0.05, "high": 0.95, "scale": "linear"},
    "user_attrs_decision_threshold": {"type": "float", "low": 0.01, "high": 0.99, "scale": "linear"},
    "params_selftrain_threshold": {"type": "float", "low": 0.6, "high": 0.95, "scale": "linear"},
    "params_scale_pos_weight": {"type": "float", "low": 0.5, "high": 5.0, "scale": "log"},  # only when resampling==none

    # numeric ints
    "params_n_estimators": {"type": "int", "low": 200, "high": 1000, "step": 100},
    "params_max_depth": {"type": "int", "low": 3, "high": 12, "step": 1},
    "params_max_delta_step": {"type": "int", "low": 0, "high": 10, "step": 1},
    "params_selftrain_max_iter": {"type": "int", "low": 5, "high": 12, "step": 1},
}

# 4) Build domains for discrete/encoded cols from TRAIN (robust to your encoding)
domains_cat = {}
for c, meta in SPEC.items():
    if meta.get("type") == "cat" and c in X_train.columns:
        domains_cat[c] = sorted(pd.Series(X_train[c]).dropna().unique().tolist())

# enforce exact missing-rate grid (matches your objective)
if "user_attrs_missing_rate" in X_train.columns:
    domains_cat["user_attrs_missing_rate"] = [0.00, 0.02, 0.05, 0.08, 0.10]

# 5) Build numeric pbounds from SPEC (not from observed min/max)
pbounds_num = {}
int_steps = {}
log_scaled = set()

for c, meta in SPEC.items():
    if c not in X_train.columns:
        continue
    t = meta.get("type")
    if t in ("float", "int"):
        pbounds_num[c] = (meta["low"], meta["high"])
        if t == "int":
            int_steps[c] = int(meta.get("step", 1))
        if meta.get("scale") == "log":
            log_scaled.add(c)

# 6) Quick report
print("X_train shape:", X_train.shape, "| y_train distribution:", y_train.value_counts(dropna=False).to_dict())

print("\nDiscrete/encoded domains (use these instead of min/max):")
for c in sorted(domains_cat):
    print(f"  - {c}: {domains_cat[c]}")

print("\nNumeric pbounds (ranges):")
for c in sorted(pbounds_num):
    lo, hi = pbounds_num[c]
    tags = []
    if c in int_steps: tags.append(f"int step={int_steps[c]}")
    if c in log_scaled: tags.append("log-scale")
    tag = f" ({', '.join(tags)})" if tags else ""
    print(f"  - {c}: ({lo}, {hi}){tag}")

X_train shape: (8861, 27) | y_train distribution: {0: 6083, 2: 1516, 3: 986, 1: 276}

Discrete/encoded domains (use these instead of min/max):
  - params_booster: [0, 1]
  - params_grow_policy: [0, 1]
  - params_normalize_type: [0, 1, 2]
  - params_sample_type: [0, 1, 2]
  - soft_decision_speed: [0, 1, 2]
  - soft_risk_preference: [0, 1, 2]
  - user_attrs_missing_rate: [0.0, 0.02, 0.05, 0.08, 0.1]
  - user_attrs_resampling_strategy: [0, 1, 2]

Numeric pbounds (ranges):
  - params_alpha: (0.0001, 100.0) (log-scale)
  - params_colsample_bytree: (0.2, 1.0)
  - params_gamma: (0.0, 10.0)
  - params_lambda: (0.0001, 100.0) (log-scale)
  - params_learning_rate: (0.0001, 0.5) (log-scale)
  - params_max_delta_step: (0, 10) (int step=1)
  - params_max_depth: (3, 12) (int step=1)
  - params_min_child_weight: (0.1, 100.0) (log-scale)
  - params_n_estimators: (200, 1000) (int step=100)
  - params_rate_drop: (0.0, 0.5)
  - params_scale_pos_weight: (0.5, 5.0) (log-scale)
  - params_selftrain_max_iter

In [13]:
# === Mutual-Information graph (works with mixed numeric + encoded categoricals) ===

from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler

# Use your X_train from previous cell
Xg = X_train.copy()

# Ensure all are numeric (encoded categoricals already are; drop non-numeric if any)
Xg = Xg.select_dtypes(include=[np.number]).copy()

# Fill missing (MI can't handle NaNs)
Xg = Xg.fillna(Xg.median(numeric_only=True))

# Standardize numeric scale (helps MI for continuous features)
Xs = pd.DataFrame(StandardScaler().fit_transform(Xg), columns=Xg.columns, index=Xg.index)

cols = list(Xs.columns)
p = len(cols)

# Mutual information score: MI(X_j -> X_i) approximated by MI(X_i, X_j) controlling nothing
# We'll build a dense symmetric score matrix first, then orient it using a simple order heuristic.
MI = np.zeros((p, p), dtype=float)
for i, target in enumerate(cols):
    y_ = Xs[target].values
    X_ = Xs.drop(columns=[target]).values
    mi = mutual_info_regression(X_, y_, discrete_features=False, random_state=0)
    # map back into full matrix
    k = 0
    for j, src in enumerate(cols):
        if src == target:
            continue
        MI[i, j] = mi[k]
        k += 1

# Build an order heuristic: "more explanatory power of others" -> later in order
# score_in = total MI into node; earlier nodes have lower incoming dependency
score_in = MI.sum(axis=1)
causal_order = list(np.argsort(score_in))  # indices

# Orient edges according to order: only allow edges from earlier -> later
adjacency_matrix = np.zeros((p, p), dtype=float)
pos = {idx: r for r, idx in enumerate(causal_order)}
for i in range(p):
    for j in range(p):
        if i == j:
            continue
        # edge j -> i allowed only if j earlier than i in order
        if pos[j] < pos[i]:
            adjacency_matrix[i, j] = MI[i, j]

# Optional: sparsify (keep only top-k parents per node)
TOPK = 5
for i in range(p):
    parents = np.argsort(adjacency_matrix[i, :])[::-1]
    keep = parents[:TOPK]
    mask = np.ones(p, dtype=bool)
    mask[keep] = False
    adjacency_matrix[i, mask] = 0.0
    adjacency_matrix[i, i] = 0.0

# Outputs in the same format you want:
print("adjacency_matrix shape:", adjacency_matrix.shape)
print("causal_order (feature indices):", causal_order)

# Helpful mapping to names
feature_names = cols
print("\nCausal order (names):")
print([feature_names[i] for i in causal_order])


adjacency_matrix shape: (27, 27)
causal_order (feature indices): [4, 25, 7, 5, 8, 19, 3, 2, 20, 10, 9, 0, 16, 6, 26, 18, 12, 17, 1, 21, 23, 14, 22, 11, 15, 13, 24]

Causal order (names):
['params_grow_policy', 'soft_risk_preference', 'params_max_delta_step', 'params_lambda', 'params_max_depth', 'user_attrs_resampling_strategy', 'params_gamma', 'params_colsample_bytree', 'user_attrs_sample_rows', 'params_n_estimators', 'params_min_child_weight', 'params_alpha', 'params_subsample', 'params_learning_rate', 'soft_decision_speed', 'user_attrs_feature_fraction', 'params_scale_pos_weight', 'user_attrs_decision_threshold', 'params_booster', 'params_selftrain_max_iter', 'user_attrs_labeled_ratio', 'params_sample_type', 'params_selftrain_threshold', 'params_normalize_type', 'params_skip_drop', 'params_rate_drop', 'user_attrs_missing_rate']


In [15]:
# --- 1) pbounds for ALL features in the exact feature order ---
features_order = X_train.columns.tolist()

# ACFX expects numeric bounds; for encoded categoricals we use min/max over the codes.
pbounds = {c: (float(X_train[c].min()), float(X_train[c].max())) for c in features_order}

# --- 2) Fit model explicitly (keeps behavior predictable) ---
model.fit(X_train, y_train)

# --- 3) Fit ACFX explainer (matrix/order must correspond to features_order) ---
# If your adjacency_matrix was computed on Xg/Xs columns, ensure it matches features_order exactly.
assert adjacency_matrix.shape == (len(features_order), len(features_order))
assert set(causal_order) == set(range(len(features_order)))

explainer.fit(
    X=X_train,
    adjacency_matrix=adjacency_matrix,
    causal_order=causal_order,
    pbounds=pbounds,
    y=y_train,
    features_order=features_order,
)

# --- 4) Helpers to snap to valid discrete/integer constraints ---
def snap_to_domain(val, domain):
    # domain is a sorted list of allowed values
    dom = np.asarray(domain, dtype=float)
    return float(dom[np.argmin(np.abs(dom - float(val)))])

def postprocess_cf(x_cf, domains_cat, int_steps):
    x = x_cf.copy()

    # snap encoded/discrete variables
    for c, dom in domains_cat.items():
        if c in x.index:
            x[c] = snap_to_domain(x[c], dom)

    # enforce integer + step constraints
    for c, step in int_steps.items():
        if c in x.index:
            v = int(round(float(x[c])))
            if step > 1:
                # snap to nearest multiple of step within bounds
                lo, hi = pbounds[c]
                grid = np.arange(int(lo), int(hi) + 1, step)
                x[c] = int(grid[np.argmin(np.abs(grid - v))])
            else:
                x[c] = v

    return x

# --- 5) Generate a counterfactual for one query ---
query_df = X_test.iloc[[0]]  # keep as DataFrame (1 row)
orig_pred = model.predict(query_df)[0]

# desired_class: choose what you need (example: flip class)
desired_class = 1 - int(orig_pred)

cf = explainer.counterfactual(
    desired_class=desired_class,
    query_instance=query_df.values[0],
)

# ACFX may return different structures; handle common cases:
if isinstance(cf, dict) and "counterfactual" in cf:
    cf_vec = pd.Series(cf["counterfactual"], index=features_order)
elif isinstance(cf, (np.ndarray, list)):
    cf_vec = pd.Series(np.asarray(cf).ravel(), index=features_order)
else:
    # if cf is already a vector-like
    cf_vec = pd.Series(cf, index=features_order)

cf_vec_pp = postprocess_cf(cf_vec, domains_cat=domains_cat, int_steps=int_steps)

print("Original pred:", int(orig_pred), "Desired:", int(desired_class))
print("\nCounterfactual (raw):")
print(cf_vec.head(10))
print("\nCounterfactual (postprocessed):")
print(cf_vec_pp.head(10))


INFO:interpret.utils._compressed_dataset:Creating native dataset
INFO:interpret.utils._compressed_dataset:Creating native dataset
INFO:interpret.glassbox._ebm._ebm:Estimating with FAST
INFO:interpret.glassbox._ebm._boost:Start boosting
INFO:interpret.utils._native:Booster allocation start
INFO:interpret.utils._native:Booster allocation end
INFO:interpret.utils._native:Deallocation boosting start
INFO:interpret.utils._native:Deallocation boosting end
INFO:interpret.utils._compressed_dataset:Creating native dataset
INFO:interpret.utils._compressed_dataset:Creating native dataset
INFO:interpret.glassbox._ebm._ebm:Estimating with FAST
INFO:interpret.glassbox._ebm._boost:Start boosting
INFO:interpret.utils._native:Booster allocation start
INFO:interpret.utils._native:Booster allocation end
INFO:interpret.utils._native:Deallocation boosting start
INFO:interpret.utils._native:Deallocation boosting end


Sampling from model... Already sampled 5 from 178 possible in data sampling...
Original pred: 0 Desired: 1

Counterfactual (raw):
params_alpha               93.348950
params_booster              1.000000
params_colsample_bytree     0.511081
params_gamma                5.977910
params_grow_policy          1.000000
params_lambda               0.000102
params_learning_rate        0.001813
params_max_delta_step       9.629178
params_max_depth           11.000000
params_min_child_weight    99.851972
dtype: float64

Counterfactual (postprocessed):
params_alpha               93.348950
params_booster              1.000000
params_colsample_bytree     0.511081
params_gamma                5.977910
params_grow_policy          1.000000
params_lambda               0.000102
params_learning_rate        0.001813
params_max_delta_step      10.000000
params_max_depth           11.000000
params_min_child_weight    99.851972
dtype: float64


In [16]:
# === Show full counterfactual (all parameters) ===
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Raw CF (all features):")
print(cf_vec)

print("\nPostprocessed CF (all features):")
print(cf_vec_pp)


Raw CF (all features):
params_alpha                       93.348950
params_booster                      1.000000
params_colsample_bytree             0.511081
params_gamma                        5.977910
params_grow_policy                  1.000000
params_lambda                       0.000102
params_learning_rate                0.001813
params_max_delta_step               9.629178
params_max_depth                   11.000000
params_min_child_weight            99.851972
params_n_estimators               400.000000
params_normalize_type               0.000000
params_scale_pos_weight             0.607529
params_rate_drop                    0.000000
params_sample_type                  0.000000
params_skip_drop                    0.000000
params_subsample                    0.462594
user_attrs_decision_threshold       0.300120
user_attrs_feature_fraction         0.664066
user_attrs_resampling_strategy      1.841386
user_attrs_sample_rows              0.500982
params_selftrain_max_iter       

In [21]:
# ===== User settings =====
DESIRED_PERF_CLASS = 3
FIXED_SOFT = {"soft_decision_speed": 2, "soft_risk_preference": 2}
QUERY_INDEX = 0
# =========================

features_order = X_train.columns.tolist()
pbounds = {c: (float(X_train[c].min()), float(X_train[c].max())) for c in features_order}

def extract_cf_vector(cf_obj, features_order):
    if isinstance(cf_obj, dict) and "counterfactual" in cf_obj:
        return pd.Series(np.asarray(cf_obj["counterfactual"]).ravel(), index=features_order)
    if isinstance(cf_obj, (np.ndarray, list, tuple)):
        return pd.Series(np.asarray(cf_obj).ravel(), index=features_order)
    return pd.Series(cf_obj, index=features_order)

def snap_to_domain(val, domain):
    dom = np.asarray(domain, dtype=float)
    return float(dom[np.argmin(np.abs(dom - float(val)))])

def postprocess_once(cf_vec, domains_cat, int_steps, pbounds, fixed_soft):
    x = cf_vec.copy()

    # force fixed soft knobs
    for k, v in fixed_soft.items():
        if k in x.index:
            x[k] = v

    # snap encoded/discrete features to valid codes
    for c, dom in domains_cat.items():
        if c in x.index and len(dom) > 0:
            x[c] = snap_to_domain(x[c], dom)

    # enforce integer + step constraints
    for c, step in int_steps.items():
        if c in x.index:
            lo, hi = pbounds[c]
            if step > 1:
                grid = np.arange(int(lo), int(hi) + 1, step)
                x[c] = int(grid[np.argmin(np.abs(grid - float(x[c])))])
            else:
                x[c] = int(round(float(x[c])))

    # clamp to bounds
    for c, (lo, hi) in pbounds.items():
        if c in x.index:
            x[c] = float(np.clip(float(x[c]), lo, hi))

    # re-apply fixed soft knobs (guarantee)
    for k, v in fixed_soft.items():
        if k in x.index:
            x[k] = v

    return x

# ---- One attempt only ----
q_df = X_test.iloc[[QUERY_INDEX]]
orig_pred = int(model.predict(q_df)[0])

cf_obj = explainer.counterfactual(
    desired_class=int(DESIRED_PERF_CLASS),
    query_instance=q_df.values[0],
)

cf_vec = extract_cf_vector(cf_obj, features_order)
cf_vec_pp = postprocess_once(cf_vec, domains_cat, int_steps, pbounds, FIXED_SOFT)

print("Original predicted class:", orig_pred)
print("Desired class:", int(DESIRED_PERF_CLASS))
print("Fixed soft:", FIXED_SOFT)

print("\nCounterfactual (all features, raw):")
print(cf_vec.to_string())

print("\nCounterfactual (all features, postprocessed):")
print(cf_vec_pp.to_string())

Sampling from model... Already sampled 5 from 1035 possible in data sampling...
Original predicted class: 0
Desired class: 3
Fixed soft: {'soft_decision_speed': 2, 'soft_risk_preference': 2}

Counterfactual (all features, raw):
params_alpha                       10.544772
params_booster                      0.836334
params_colsample_bytree             0.586692
params_gamma                        3.504963
params_grow_policy                  0.883529
params_lambda                       2.352747
params_learning_rate                0.073302
params_max_delta_step               4.750229
params_max_depth                   11.820390
params_min_child_weight            18.896802
params_n_estimators               400.575705
params_normalize_type               0.909530
params_scale_pos_weight             4.777185
params_rate_drop                    0.000716
params_sample_type                  0.104457
params_skip_drop                    0.594325
params_subsample                    0.538600
user_at

In [22]:
import json


# ===== Inputs =====
MAPS_JSON = "data/acfx_maps.json"
OUT_CSV   = "data/acfx_cfs_decoded.csv"

# Provide ONE of these:
#  - cf_vec_pp (pd.Series indexed by feature names), OR
#  - cf_df (pd.DataFrame of multiple CFs, columns are features)
# The cell will detect what you have.
# ==================

with open(MAPS_JSON, "r", encoding="utf-8") as f:
    maps = json.load(f)

CAT_COLS = list(maps["categorical"].keys())  # same categorical columns used in your encoding
bins   = maps["target"]["bins"]
labels = maps["target"]["labels"]

def _decode_categorical(col_name: str, values: pd.Series) -> pd.Series:
    cats = maps["categorical"][col_name]
    # ACFX returns floats -> snap to nearest int code, clamp to valid range
    codes = np.rint(values.astype(float)).astype(int)
    codes = np.clip(codes, 0, len(cats) - 1)
    return pd.Series([cats[i] for i in codes], index=values.index)

def decode_acfx_counterfactuals(cf_encoded: pd.DataFrame, desired_class: int = None) -> pd.DataFrame:
    df = cf_encoded.copy()

    # Decode categoricals
    for c in CAT_COLS:
        if c in df.columns:
            df[c] = _decode_categorical(c, df[c])

    # Enforce the same DART conditionals as in your encoding pipeline
    if "params_booster" in df.columns:
        not_dart = df["params_booster"].astype(str) != "dart"

        for c in ["params_rate_drop", "params_skip_drop"]:
            if c in df.columns:
                df.loc[not_dart, c] = 0.0

        for c in ["params_sample_type", "params_normalize_type"]:
            if c in df.columns:
                df.loc[not_dart, c] = "NA"

    # Make some known ints look like ints (optional, purely cosmetic)
    for c in ["params_n_estimators", "params_max_depth", "params_max_delta_step", "params_selftrain_max_iter"]:
        if c in df.columns:
            df[c] = np.rint(df[c].astype(float)).astype(int)

    # Add a performance band column that matches your cat_value binning
    # If you pass desired_class, we can attach its band. Otherwise we skip.
    if desired_class is not None:
        # labels are [0,1,2,3] and bins are [0,0.5,0.6,0.7,1.0]
        idx = labels.index(int(desired_class)) if int(desired_class) in labels else None
        if idx is not None:
            lo, hi = bins[idx], bins[idx + 1]
            df["cat_value_target"] = int(desired_class)
            df["f1_band"] = f"[{lo:.2f}, {hi:.2f}]"
            df["f1_midpoint"] = (lo + hi) / 2.0

    return df

# ---- Get CFs in a DataFrame form ----
if "cf_df" in globals() and isinstance(cf_df, pd.DataFrame):
    cf_encoded_df = cf_df.copy()
elif "cf_vec_pp" in globals() and isinstance(cf_vec_pp, pd.Series):
    cf_encoded_df = cf_vec_pp.to_frame().T
else:
    raise RuntimeError("Provide cf_df (DataFrame) or cf_vec_pp (Series) before running this cell.")

# Desired class: use your variable if present
desired_cls = int(DESIRED_PERF_CLASS) if "DESIRED_PERF_CLASS" in globals() else None

# ---- Decode + export ----
cf_decoded_df = decode_acfx_counterfactuals(cf_encoded_df, desired_class=desired_cls)

cf_decoded_df.to_csv(OUT_CSV, index=False)

print("Decoded CFs saved to:", OUT_CSV)
display(cf_decoded_df.head(10))


Decoded CFs saved to: data/acfx_cfs_decoded.csv


,params_alpha,params_booster,params_colsample_bytree,params_gamma,params_grow_policy,params_lambda,params_learning_rate,params_max_delta_step,params_max_depth,params_min_child_weight,params_n_estimators,params_normalize_type,params_scale_pos_weight,params_rate_drop,params_sample_type,params_skip_drop,params_subsample,user_attrs_decision_threshold,user_attrs_feature_fraction,user_attrs_resampling_strategy,user_attrs_sample_rows,params_selftrain_max_iter,params_selftrain_threshold,user_attrs_labeled_ratio,user_attrs_missing_rate,soft_risk_preference,soft_decision_speed,cat_value_target,f1_band,f1_midpoint
0,0.000102,gbtree,0.511081,0.011017,lossguide,0.000108,0.498598,3,12,0.100616,700,NA,0.550304,0.000000,NA,0.0,0.999332,0.685961,0.998999,none,0.999902,0,0.990778,0.978913,0.0,recall_pref,slow,3,"[0.70, 1.00]",0.85
1,0.000306,gbtree,0.475923,0.542617,depthwise,0.183407,0.046004,0,9,0.256119,1000,NA,3.876156,0.000000,NA,0.0,0.985403,0.646175,0.928778,none,0.715005,0,1.000000,1.000000,0.0,recall_pref,slow,3,"[0.70, 1.00]",0.85
2,0.000102,gbtree,0.976045,5.977910,lossguide,0.000109,0.498557,3,11,0.100147,400,NA,0.549883,0.000000,NA,0.0,0.999134,0.685895,0.998947,none,0.999840,2,0.927031,0.975717,0.0,recall_pref,slow,3,"[0.70, 1.00]",0.85
3,0.001614,dart,0.511081,0.011395,lossguide,0.000109,0.499448,0,11,0.100541,700,NA,0.607529,0.008668,NA,0.0,0.462594,0.653805,0.999079,undersample,0.810662,2,0.927031,0.986317,0.1,recall_pref,slow,3,"[0.70, 1.00]",0.85
4,0.001614,dart,0.511081,0.012758,lossguide,0.000108,0.499016,0,11,1.726523,400,forest,0.549477,0.008968,weighted,0.0,0.999533,0.685936,0.999096,undersample,0.810662,2,0.996966,0.979658,0.0,recall_pref,slow,3,"[0.70, 1.00]",0.85
